## InMemoryVectorStore（了解即可）

### 1. 文档加载

In [1]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("../data/deepseek百度百科.txt", encoding="utf-8")
docs = loader.load()
print(f"文档数量{len(docs)}")
for i, doc in enumerate(docs):
    print(f"================Doc_{i}.metadata=====================")
    print("metadata = ", doc.metadata)
    print(f"================Doc_{i}.page_content=====================")
    print("page_content = ", doc.page_content)

文档数量1
================Doc_0.metadata=====================
metadata =  {'source': '../data/deepseek百度百科.txt'}
================Doc_0.page_content=====================
page_content =  DeepSeek，全称杭州深度求索人工智能基础技术研究有限公司。DeepSeek是一家创新型科技公司，成立于2023年7月17日 ，使用数据蒸馏技术，得到更为精炼、有用的数据。由知名私募巨头幻方量化孕育而生，专注于开发先进的大语言模型（LLM）和相关技术。注册地址：浙江省杭州市拱墅区环城北路169号汇金国际大厦西1幢1201室。法定代表人为裴湉，经营范围包括技术服务、技术开发、软件开发等。
2024年1月5日至6月，相继发布DeepSeek LLM、DeepSeek-Coder、DeepSeekMath、DeepSeek-VL、DeepSeek-V2、DeepSeek-Coder-V2模型。 2024年9月5日，更新API支持文档，宣布合并DeepSeek Coder V2和DeepSeek V2 Chat，推出DeepSeek V2.5。12月13日，发布DeepSeek-VL2。12月26日，正式上线DeepSeek-V3首个版本并同步开源。2025年1月31日，英伟达宣布DeepSeek-R1模型登陆NVIDIANIM。同一时段内，亚马逊和微软也接入DeepSeek-R1模型。 2月5日，DeepSeek-R1、V3、Coder等系列模型，已陆续上线国家超算互联网平台。2月6日消息，澳大利亚政府以所谓“担心安全风险”为由，已禁止在所有政府设备中使用DeepSeek。2月8日，DeepSeek正式登陆苏州，并在苏州市公共算力服务平台上完成部署上线，为用户提供开箱即用的软硬件一体服务。 截至2月9日，DeepSeek App的累计下载量已超1.1亿次，周活跃用户规模最高近9700万。 2月20日消息，已有超200家企业宣布接入DeepSeek。 2月26日，DeepSeek宣布开源DeepGEMM。

发展历程
早期发展
DeepSeek成立于2023年7月17日，由知名量化资管巨头幻方量化创立。DeepSe

### 2. 文档切分

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    separators=[
        "\n\n",   # 段落边界，优先保留
        "\n",     # 换行/小节边界
        "。",     # 句子结束
        "！",     # 感叹句结束
        "？",     # 疑问句结束
        "；",     # 分句边界
        "，",     # 短语边界
        "、",     # 并列词边界
        " ",      # 中英混排兜底
        "",       # 最后兜底：按字符切
    ],
    chunk_size=100,
    chunk_overlap=10,
    is_separator_regex=False,       # 告诉分割器上面包含正则
)
chunks = splitter.split_documents(docs)
print(f"原始文本长度：{len(docs)} 字符")
print(f"切分后的块数：{len(chunks)} 个块")
for i, chunk in enumerate(chunks):
    print(f"--------Chunk {i+1} {len(chunk.page_content)} 个字符---------")
    print(chunk)
    print()

原始文本长度：1 字符
切分后的块数：338 个块
--------Chunk 1 87 个字符---------
page_content='DeepSeek，全称杭州深度求索人工智能基础技术研究有限公司。DeepSeek是一家创新型科技公司，成立于2023年7月17日 ，使用数据蒸馏技术，得到更为精炼、有用的数据' metadata={'source': '../data/deepseek百度百科.txt'}

--------Chunk 2 77 个字符---------
page_content='。由知名私募巨头幻方量化孕育而生，专注于开发先进的大语言模型（LLM）和相关技术。注册地址：浙江省杭州市拱墅区环城北路169号汇金国际大厦西1幢1201室' metadata={'source': '../data/deepseek百度百科.txt'}

--------Chunk 3 32 个字符---------
page_content='。法定代表人为裴湉，经营范围包括技术服务、技术开发、软件开发等。' metadata={'source': '../data/deepseek百度百科.txt'}

--------Chunk 4 12 个字符---------
page_content='2024年1月5日至6月' metadata={'source': '../data/deepseek百度百科.txt'}

--------Chunk 5 89 个字符---------
page_content='，相继发布DeepSeek LLM、DeepSeek-Coder、DeepSeekMath、DeepSeek-VL、DeepSeek-V2、DeepSeek-Coder-V2模型' metadata={'source': '../data/deepseek百度百科.txt'}

--------Chunk 6 98 个字符---------
page_content='。 2024年9月5日，更新API支持文档，宣布合并DeepSeek Coder V2和DeepSeek V2 Chat，推出DeepSeek V2.5。12月13日，发布DeepSeek-VL2' metadata={'source': '../data/deepseek百度百科.txt'

### 3. 文档向量化

In [3]:
from langchain_ollama import OllamaEmbeddings

# 1.初始化向量模型
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)

### 4. VectorStore

In [4]:
from langchain_core.vectorstores import InMemoryVectorStore
# 2.初始化向量数据库
vector_store = InMemoryVectorStore(embedding=ollama_embeddings)

#### 4.1 添加/删除文档
- `add_documents`：接收 `list[Document]`，可以批量添加文档
- `delete`：接收 `list[str]`，也就是 id 集合，可以根据 id 批量删除文档

In [5]:
# 保存文档到向量数据库
ids = vector_store.add_documents(chunks)
print(ids)

['256e4db2-7a1f-4b70-96b3-4809f0256de3', '2c04ac25-9840-43e7-aab9-061a3e9b12cb', 'ef141e06-9e4c-445b-9acd-69f36cc8bc7b', '0b7e7eed-e6d5-4e45-a6b5-1e5fca62e401', '3f48c98d-d6b2-4bae-accf-0b163fdc93d7', '59828a0d-4fff-49c8-afcd-7139357e3fb5', 'f84d30bf-6610-40a3-b6f5-2077107e99d5', 'e8e92cef-2f63-4fb4-91ed-8b8f76c21788', '0aca9a7b-1967-43d8-ae5c-e08080ebaec1', '6eec83c1-d30d-4acb-8dfa-83a811e0f554', '9f706925-3cff-4320-a6eb-b621d54d51ff', '7cd304de-8656-4fab-be7a-736652aed319', '4ead435b-7e15-4883-8a9d-38c69f0b204e', '69c8f4dc-11e4-4b81-9856-54d7c442597a', '05475f36-fc82-4805-9b26-d9eae0728337', 'd801b66c-c52e-4ca9-a4aa-0e943ad55c56', 'b1abf67e-e931-4a25-ae7e-7a8523b661ca', '46373e53-1f1a-45ca-a86d-b2728d04f63d', '6f6173db-bed7-419a-82df-5fc683b80408', '81312e1a-d338-4655-84a9-aff89c3fff9b', '6e132a28-50d8-4647-a738-6e6c8825afde', 'c434801c-093b-40f9-80fc-66a06c8cbde0', 'e6612299-c445-4371-bf73-4562c503a931', 'b35434d0-ed4e-4728-a925-d8bf475fad22', 'd5cd9daf-6973-4f3e-87e4-f0263f4db5d7',

In [6]:
# 删除旧文档
vector_store.delete(["doc_id1", "doc_id2"])

#### 4.2 检索文档
VectorStore提供了多个检索文档的方法，例如：
- `search`: 通用搜索方法，支持最多样化的参数
- `similarity_search`: 基于相似度的搜索
- `similarity_search_with_relevance_scores`: 基于相似度搜索，并且会返回相似度得分

`similarity_search` 方法实现文档相似度检索，其核心参数为：
- `query`: 查询条件（用户的问题）
其它参数(并不是所有向量库都支持):
- `k`: 要返回的文档最大数量（默认值：4）
- `score_threshold`: 相似度打分的最小阈值，低于这个分值的文档会被丢弃
- `filter`: 按文档元数据（metadata）筛选

相似度检索



In [7]:
""" similarity_search 只返回文档，没有得分。返回值类型是list[Document]"""
# 用户问题
query = "deepseek有什么危险"

# 相似度检索，返回值是list[Document]
results = vector_store.similarity_search(
    query=query,
    k = 5,
)

# 打印结果
for doc in results:
    print(doc.model_dump_json(indent=2))
    print("===========================")

{
  "id": "0b9b2b8b-7069-4aec-b17f-80c0d1a5429c",
  "metadata": {
    "source": "../data/deepseek百度百科.txt"
  },
  "page_content": "DeepSeek引发全球轰动和一些人的焦虑恐慌，说明技术遏制和技术限制无法奏效，这是全世界、特别是美国需要学习的一课。（中国常驻联合国代表傅聪评）",
  "type": "Document"
}
{
  "id": "7e51e4ba-05d2-4f6a-9d5d-174158ed8dfd",
  "metadata": {
    "source": "../data/deepseek百度百科.txt"
  },
  "page_content": "DeepSeek：这锅得背！技术越强，责任越大",
  "type": "Document"
}
{
  "id": "a0df5718-c8d2-4f09-b54b-58ae3955ea55",
  "metadata": {
    "source": "../data/deepseek百度百科.txt"
  },
  "page_content": "。DeepSeek被疑过度收集个人信息，且尚未回复韩国个人信息保护委员会的质询函，依然存在所谓“安全隐患”。",
  "type": "Document"
}
{
  "id": "def522cc-d385-4376-8b69-1dbeec2864db",
  "metadata": {
    "source": "../data/deepseek百度百科.txt"
  },
  "page_content": "。本部表示，根据政府方针，校方先对DeepSeek平台采取了屏蔽措施，直至其个人信息保护等安全性得到确认。以学术研究或教育为目的使用DeepSeek需提前申请",
  "type": "Document"
}
{
  "id": "c2301873-59c2-4ae9-9f0a-81c16a1c7ba8",
  "metadata": {
    "source": "../data/deepseek百度百科.txt"
  },
  "page_content": "DeepSeek可能是个

带相似度得分的检索

In [8]:
""" similarity_search_with_score 相似度搜索并返回分数，结果是list[tuple[Document, float]]"""
results = vector_store.similarity_search_with_score(
    query,
    k = 5,
)

# 遍历结果，
for doc, score in results:
    print(f"==========score: {score}=============")
    print(doc.model_dump_json(indent=2))

==========score: 0.7597182611089963=============
{
  "id": "0b9b2b8b-7069-4aec-b17f-80c0d1a5429c",
  "metadata": {
    "source": "../data/deepseek百度百科.txt"
  },
  "page_content": "DeepSeek引发全球轰动和一些人的焦虑恐慌，说明技术遏制和技术限制无法奏效，这是全世界、特别是美国需要学习的一课。（中国常驻联合国代表傅聪评）",
  "type": "Document"
}
==========score: 0.7597123918539913=============
{
  "id": "7e51e4ba-05d2-4f6a-9d5d-174158ed8dfd",
  "metadata": {
    "source": "../data/deepseek百度百科.txt"
  },
  "page_content": "DeepSeek：这锅得背！技术越强，责任越大",
  "type": "Document"
}
==========score: 0.7595388185584798=============
{
  "id": "a0df5718-c8d2-4f09-b54b-58ae3955ea55",
  "metadata": {
    "source": "../data/deepseek百度百科.txt"
  },
  "page_content": "。DeepSeek被疑过度收集个人信息，且尚未回复韩国个人信息保护委员会的质询函，依然存在所谓“安全隐患”。",
  "type": "Document"
}
==========score: 0.7586440661467102=============
{
  "id": "def522cc-d385-4376-8b69-1dbeec2864db",
  "metadata": {
    "source": "../data/deepseek百度百科.txt"
  },
  "page_content": "。本部表示，根据政府方针，校方先对DeepSeek平台采取了屏蔽措施，直至其个人信息保护等安全性得到确认